In [ ]:
%sql
SHOW DATABASES;

In [ ]:
%sql
USE CATALOG ipl_matchday_360;
USE SCHEMA default;

In [ ]:
%sql
SHOW TABLES;

In [ ]:
%sql
SHOW CATALOGS;

In [ ]:
%sql
SHOW SCHEMAS IN ipl_matchday_360;

In [ ]:
%sql
USE CATALOG ipl_matchday_360;

In [ ]:
%sql
USE SCHEMA default;

In [ ]:
%sql
SHOW TABLES;

In [ ]:
%sql
SHOW TABLES IN workspace.default;

In [ ]:
%sql
DESCRIBE workspace.default.silver_matches_candidate;

In [ ]:
%sql
SELECT *
FROM workspace.default.silver_matches_candidate
LIMIT 10;

In [ ]:
%sql
DESCRIBE workspace.default.silver_deliveries_candidate;

In [ ]:
%sql
SELECT *
FROM workspace.default.silver_deliveries_candidate
LIMIT 10;

In [ ]:
%sql
DESCRIBE workspace.default.silver_players_candidate;

In [ ]:
%sql
SELECT *
FROM workspace.default.silver_players_candidate
LIMIT 10;

In [ ]:
%sql
DESCRIBE workspace.default.silver_venues_candidate;

In [ ]:
%sql
SELECT *
FROM workspace.default.silver_venues_candidate
LIMIT 10;

In [ ]:
%sql
SELECT 'matches' AS table_name, COUNT(*) AS record_count
FROM workspace.default.silver_matches_candidate

UNION ALL

SELECT 'deliveries', COUNT(*)
FROM workspace.default.silver_deliveries_candidate

UNION ALL

SELECT 'players', COUNT(*)
FROM workspace.default.silver_players_candidate

UNION ALL

SELECT 'venues', COUNT(*)
FROM workspace.default.silver_venues_candidate;

In [ ]:
%sql
SELECT COUNT(*) AS matches
FROM workspace.default.silver_matches_candidate;

In [ ]:
%sql
SELECT COUNT(*) AS deliveries
FROM workspace.default.silver_deliveries_candidate;

In [ ]:
%sql
SELECT COUNT(*) AS players
FROM workspace.default.silver_players_candidate;

In [ ]:
%sql
SELECT COUNT(*) AS venues
FROM workspace.default.silver_venues_candidate;

In [ ]:
%sql
DESCRIBE workspace.default.silver_matches_candidate;

In [ ]:
%sql
DESCRIBE workspace.default.silver_deliveries_candidate;

In [ ]:
%sql
DESCRIBE workspace.default.silver_players_candidate;

In [ ]:
%sql
DESCRIBE workspace.default.silver_venues_candidate;

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
matches = spark.table("workspace.default.silver_matches_candidate")
deliveries = spark.table("workspace.default.silver_deliveries_candidate")
players = spark.table("workspace.default.silver_players_candidate")
venues = spark.table("workspace.default.silver_venues_candidate")

print("Matches:", matches.count())
print("Deliveries:", deliveries.count())
print("Players:", players.count())
print("Venues:", venues.count())

In [ ]:
print("MATCH COLUMNS:")
print(matches.columns)

print("\nDELIVERY COLUMNS:")
print(deliveries.columns)

print("\nPLAYER COLUMNS:")
print(players.columns)

print("\nVENUE COLUMNS:")
print(venues.columns)

DQ-IPL-001 for MATCHES

In [ ]:
match_dup = (
    matches
    .groupBy("match_id")
    .count()
    .filter(F.col("count") > 1)
    .select("match_id")
    .withColumn("is_duplicate", F.lit(True))
)

In [ ]:
matches_q = (
    matches
    .join(match_dup, on="match_id", how="left")
    .withColumn(
        "r001",
        F.when(
            F.col("match_id").isNull() |
            (F.trim(F.col("match_id")) == "") |
            F.col("is_duplicate").isNotNull(),
            F.lit("DQ-IPL-001")
        )
    )
)

DQ-IPL-002 for MATCHES

In [ ]:
valid_venues = venues.select("venue_id").distinct()

In [ ]:
matches_q = (
    matches_q
    .join(
        valid_venues.withColumn("venue_exists", F.lit(True)),
        on="venue_id",
        how="left"
    )
    .withColumn(
        "r002",
        F.when(
            F.col("venue_id").isNotNull() &
            (F.trim(F.col("venue_id")) != "") &
            F.col("venue_exists").isNull(),
            F.lit("DQ-IPL-002")
        )
    )
)

DQ-IPL-003 for MATCHES

In [ ]:
matches_q = matches_q.withColumn(
    "r003",
    F.when(
        F.col("team_1_id").isNull() |
        F.col("team_2_id").isNull() |
        (F.col("team_1_id") == F.col("team_2_id")),
        F.lit("DQ-IPL-003")
    )
)

DQ-IPL-007 for MATCHES

In [ ]:
matches_q = matches_q.withColumn(
    "r007",
    F.when(
        (
            (F.lower(F.col("match_status")) == "completed") &
            (
                F.col("winner_id").isNull() |
                (
                    (F.col("winner_id") != F.col("team_1_id")) &
                    (F.col("winner_id") != F.col("team_2_id"))
                )
            )
        )
        |
        (
            (F.lower(F.col("match_status")) == "no_result") &
            F.col("winner_id").isNotNull()
        ),
        F.lit("DQ-IPL-007")
    )
)

Combine match rule IDs

In [ ]:
matches_q = matches_q.withColumn(
    "dq_rule_ids",
    F.concat_ws(
        ",",
        F.col("r001"),
        F.col("r002"),
        F.col("r003"),
        F.col("r007")
    )
)

add pass or fail


In [ ]:
matches_q = matches_q.withColumn(
    "dq_status",
    F.when(
        F.col("dq_rule_ids") == "",
        F.lit("PASS")
    ).otherwise(
        F.lit("FAIL")
    )
)

Add failure reasons

In [ ]:
matches_q = (
    matches_q
    .withColumn(
        "failure_reasons",
        F.concat_ws(
            "; ",
            F.when(F.col("r001").isNotNull(),
                   "Required or duplicate match_id"),
            F.when(F.col("r002").isNotNull(),
                   "venue_id does not resolve to a venue"),
            F.when(F.col("r003").isNotNull(),
                   "Match teams are invalid or identical"),
            F.when(F.col("r007").isNotNull(),
                   "Match result/winner is inconsistent")
        )
    )
    .withColumn(
        "severity",
        F.when(F.col("dq_status") == "FAIL", "ERROR")
         .otherwise("NONE")
    )
    .withColumn("record_key", F.col("match_id"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn(
        "rework_status",
        F.when(F.col("dq_status") == "FAIL", "OPEN")
         .otherwise("NOT_REQUIRED")
    )
)

check matchresult

In [ ]:
display(
    matches_q.select(
        "source_record_id",
        "match_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons",
        "severity",
        "record_key",
        "rework_status"
    )
)

In [ ]:
matches_q.groupBy("dq_status").count().show()

DQ-IPL-001 for DELIVERIES

In [ ]:
delivery_dup = (
    deliveries
    .groupBy("delivery_id")
    .count()
    .filter(F.col("count") > 1)
    .select("delivery_id")
    .withColumn("is_duplicate", F.lit(True))
)

In [ ]:
deliveries_q = (
    deliveries
    .join(delivery_dup, on="delivery_id", how="left")
    .withColumn(
        "r001",
        F.when(
            F.col("delivery_id").isNull() |
            (F.trim(F.col("delivery_id")) == "") |
            F.col("is_duplicate").isNotNull(),
            F.lit("DQ-IPL-001")
        )
    )
)

DQ-IPL-002 for DELIVERIES

In [ ]:
valid_matches = matches.select("match_id").distinct()

In [ ]:
deliveries_q = (
    deliveries_q
    .join(
        valid_matches.withColumn("match_exists", F.lit(True)),
        on="match_id",
        how="left"
    )
    .withColumn(
        "r002",
        F.when(
            F.col("match_id").isNotNull() &
            (F.trim(F.col("match_id")) != "") &
            F.col("match_exists").isNull(),
            F.lit("DQ-IPL-002")
        )
    )
)

DQ-IPL-003 for DELIVERIES

In [ ]:
delivery_team_check = (
    deliveries_q
    .join(
        matches.select(
            "match_id",
            "team_1_id",
            "team_2_id"
        ),
        on="match_id",
        how="left"
    )
)

In [ ]:
delivery_team_check = delivery_team_check.withColumn(
    "r003",
    F.when(
        (
            F.col("batting_team_id").isNotNull() &
            ~F.col("batting_team_id").isin(
                F.col("team_1_id"),
                F.col("team_2_id")
            )
        )
        |
        (
            F.col("bowling_team_id").isNotNull() &
            ~F.col("bowling_team_id").isin(
                F.col("team_1_id"),
                F.col("team_2_id")
            )
        )
        |
        (
            F.col("batting_team_id").isNotNull() &
            F.col("bowling_team_id").isNotNull() &
            (F.col("batting_team_id") == F.col("bowling_team_id"))
        ),
        F.lit("DQ-IPL-003")
    )
)

In [ ]:
deliveries_q = delivery_team_check

DQ-IPL-004

In [ ]:
deliveries_q = deliveries_q.withColumn(
    "r004_basic",
    F.when(
        (F.col("innings_no") < 1) |
        (F.col("innings_no") > 2) |
        (F.col("over_no") < 1) |
        (F.col("over_no") > 20) |
        (F.col("ball_seq") <= 0) |
        F.col("ball_seq").isNull(),
        F.lit("DQ-IPL-004")
    )
)

In [ ]:
seq_dup = (
    deliveries
    .groupBy(
        "match_id",
        "innings_no",
        "over_no",
        "ball_seq"
    )
    .count()
    .filter(F.col("count") > 1)
    .withColumn("sequence_duplicate", F.lit(True))
    .drop("count")
)

In [ ]:
deliveries_q = (
    deliveries_q
    .join(
        seq_dup,
        ["match_id", "innings_no", "over_no", "ball_seq"],
        "left"
    )
    .withColumn(
        "r004",
        F.when(
            F.col("r004_basic").isNotNull() |
            F.col("sequence_duplicate").isNotNull(),
            F.lit("DQ-IPL-004")
        )
    )
)

DQ-IPL-005

In [ ]:
deliveries_q = deliveries_q.withColumn(
    "r005",
    F.when(
        (~F.col("runs_batter").isin(0, 1, 2, 3, 4, 5, 6))
        |
        F.col("runs_batter").isNull()
        |
        (F.col("runs_extras").isNull())
        |
        (F.col("runs_extras") < 0)
        |
        (F.col("total_runs") !=
         (F.col("runs_batter") + F.col("runs_extras"))),
        F.lit("DQ-IPL-005")
    )
)

DQ-IPL-006

In [ ]:
deliveries_q = deliveries_q.withColumn(
    "r006",
    F.when(
        (
            (F.col("wicket_flag") == True) &
            (
                F.col("dismissal_type").isNull() |
                F.col("dismissed_player_id").isNull()
            )
        )
        |
        (
            (F.col("wicket_flag") == False) &
            (
                F.col("dismissal_type").isNotNull() |
                F.col("dismissed_player_id").isNotNull()
            )
        ),
        F.lit("DQ-IPL-006")
    )
)

DQ-IPL-002 player references

In [ ]:
valid_players = players.select("player_id").distinct()

In [ ]:
deliveries_q = (
    deliveries_q
    .join(
        valid_players.withColumn("batter_exists", F.lit(True)),
        deliveries_q.batter_id == valid_players.player_id,
        "left"
    )
    .drop(valid_players.player_id)
)

In [ ]:
print("DELIVERY COLUMNS:")
for c in deliveries.columns:
    print(c)

Find the declared innings-score column

In [ ]:
print("MATCH COLUMNS:")
for i, c in enumerate(matches.columns):
    print(i, c)

Implement DQ-IPL-008

In [ ]:
from pyspark.sql import functions as F

# ============================================================
# DQ-IPL-008: Innings Reconciliation
# ============================================================

# 1. Calculate runs from deliveries
innings_calculated = (
    deliveries
    .groupBy("match_id", "innings_no")
    .agg(
        F.sum("total_runs").alias("calculated_runs")
    )
)

# 2. Convert declared innings scores into rows
innings_declared = (
    matches
    .select(
        "match_id",
        F.expr("""
            stack(
                2,
                1, innings_1_runs,
                2, innings_2_runs
            )
        """).alias("innings_no", "declared_runs")
    )
)

# 3. Compare calculated runs with declared runs
dq_008_group = (
    innings_calculated
    .join(
        innings_declared,
        ["match_id", "innings_no"],
        "left"
    )
    .withColumn(
        "dq_008_status",
        F.when(
            F.col("declared_runs").isNull(),
            "FAIL"
        )
        .when(
            F.col("calculated_runs") == F.col("declared_runs"),
            "PASS"
        )
        .otherwise("FAIL")
    )
    .withColumn(
        "dq_008_reason",
        F.when(
            F.col("declared_runs").isNull(),
            "Declared innings score is missing"
        )
        .when(
            F.col("calculated_runs") != F.col("declared_runs"),
            F.concat(
                F.lit("Innings run mismatch: calculated="),
                F.col("calculated_runs"),
                F.lit(", declared="),
                F.col("declared_runs")
            )
        )
    )
)

display(dq_008_group)

In [ ]:
# ============================================================
# Propagate DQ-IPL-008 failure to affected deliveries
# ============================================================

dq_008_failed_groups = (
    dq_008_group
    .filter(F.col("dq_008_status") == "FAIL")
    .select(
        "match_id",
        "innings_no",
        "dq_008_reason"
    )
)

deliveries_dq_008 = (
    deliveries
    .join(
        dq_008_failed_groups,
        ["match_id", "innings_no"],
        "left"
    )
    .withColumn(
        "dq_008_failed",
        F.when(F.col("dq_008_reason").isNotNull(), True)
        .otherwise(False)
    )
)

display(
    deliveries_dq_008
    .filter(F.col("dq_008_failed") == True)
)

Build the final DQ result per delivery record

In [ ]:
# ============================================================
# STEP 3: Combine DQ results at delivery record level
# ============================================================

dq_results = (
    deliveries_dq_008
    .withColumn(
        "dq_rule_ids",
        F.when(
            F.col("dq_008_failed"),
            F.lit("DQ-IPL-008")
        ).otherwise(F.lit(""))
    )
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_008_failed"),
            F.lit("FAIL")
        ).otherwise(F.lit("PASS"))
    )
    .withColumn(
        "failure_reasons",
        F.col("dq_008_reason")
    )
)

display(
    dq_results.select(
        "source_record_id",
        "match_id",
        "innings_no",
        "delivery_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons"
    )
)

Combine DQ-001 to DQ-008

In [ ]:
# ============================================================
# STEP 4: Final DQ result per delivery record
# ============================================================

# Start with every delivery record
final_dq = deliveries.select(
    "source_record_id",
    "match_id",
    "delivery_id"
)

# Add DQ-008 result
final_dq = (
    final_dq
    .join(
        deliveries_dq_008.select(
            "source_record_id",
            "dq_008_failed",
            "dq_008_reason"
        ),
        "source_record_id",
        "left"
    )
)

# Replace null with PASS for DQ-008
final_dq = (
    final_dq
    .withColumn(
        "dq_008_failed",
        F.coalesce(F.col("dq_008_failed"), F.lit(False))
    )
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_008_failed") == True,
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn(
        "dq_rule_ids",
        F.when(
            F.col("dq_008_failed") == True,
            "DQ-IPL-008"
        ).otherwise(None)
    )
    .withColumn(
        "failure_reasons",
        F.col("dq_008_reason")
    )
)

display(final_dq)

Merge DQ-001 to DQ-008

In [ ]:
# Show DataFrames currently available in the notebook
for name, value in list(globals().items()):
    if name.startswith("dq"):
        print(name, type(value))

Create the final DQ framework

In [ ]:
# ============================================================
# STEP 5: Create final delivery-level DQ result
# ============================================================

final_dq = (
    deliveries
    .select(
        "source_record_id",
        "match_id",
        "delivery_id",
        "innings_no",
        "over_no",
        "ball_seq"
    )
    .join(
        deliveries_dq_008.select(
            "source_record_id",
            "dq_008_failed",
            "dq_008_reason"
        ),
        "source_record_id",
        "left"
    )
)

# Fill missing DQ-008 result
final_dq = (
    final_dq
    .withColumn(
        "dq_008_failed",
        F.coalesce(F.col("dq_008_failed"), F.lit(False))
    )
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_008_failed"),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn(
        "dq_rule_ids",
        F.when(
            F.col("dq_008_failed"),
            F.lit("DQ-IPL-008")
        ).otherwise(F.lit(""))
    )
    .withColumn(
        "failure_reasons",
        F.col("dq_008_reason")
    )
)

display(final_dq)

In [ ]:
final_dq.groupBy("dq_status").count().show()

Find your existing DQ-001 to DQ-007 result DataFrames

In [ ]:
from pyspark.sql import DataFrame

for name, value in list(globals().items()):
    if isinstance(value, DataFrame):
        print(name)

Step 7 — Inspect the existing DQ outputs

In [ ]:
# Inspect existing DQ result DataFrames

check_dfs = [
    "match_dup",
    "matches_q",
    "valid_venues",
    "delivery_dup",
    "deliveries_q",
    "valid_matches",
    "delivery_team_check",
    "seq_dup",
    "valid_players"
]

for name in check_dfs:
    df = globals().get(name)
    print("\n==============================")
    print(name)
    print("==============================")
    print(df.columns)

build the final delivery DQ result

In [ ]:
# ============================================================
# STEP 8: Final DQ result for deliveries
# ============================================================

# Start from deliveries_q, which already contains DQ-001 to DQ-006
final_deliveries_dq = (
    deliveries_q
    .join(
        deliveries_dq_008.select(
            "source_record_id",
            "dq_008_failed",
            "dq_008_reason"
        ),
        "source_record_id",
        "left"
    )
)

# Convert rule flags to integers safely
for rule in ["r001", "r002", "r003", "r004", "r005", "r006"]:
    final_deliveries_dq = final_deliveries_dq.withColumn(
        rule,
        F.coalesce(F.col(rule).cast("int"), F.lit(0))
    )

final_deliveries_dq = (
    final_deliveries_dq

    # --------------------------------------------------------
    # Combine all failed rule IDs into ONE column
    # --------------------------------------------------------
    .withColumn(
        "dq_rule_ids",
        F.concat_ws(
            ",",
            F.when(F.col("r001") == 1, "DQ-IPL-001"),
            F.when(F.col("r002") == 1, "DQ-IPL-002"),
            F.when(F.col("r003") == 1, "DQ-IPL-003"),
            F.when(F.col("r004") == 1, "DQ-IPL-004"),
            F.when(F.col("r005") == 1, "DQ-IPL-005"),
            F.when(F.col("r006") == 1, "DQ-IPL-006"),
            F.when(F.col("dq_008_failed") == True, "DQ-IPL-008")
        )
    )

    # --------------------------------------------------------
    # PASS if no rule failed
    # --------------------------------------------------------
    .withColumn(
        "dq_status",
        F.when(
            (F.col("r001") == 1) |
            (F.col("r002") == 1) |
            (F.col("r003") == 1) |
            (F.col("r004") == 1) |
            (F.col("r005") == 1) |
            (F.col("r006") == 1) |
            (F.coalesce(F.col("dq_008_failed"), F.lit(False)) == True),
            "FAIL"
        ).otherwise("PASS")
    )

    # --------------------------------------------------------
    # Failure reason
    # --------------------------------------------------------
    .withColumn(
        "failure_reasons",
        F.concat_ws(
            "; ",
            F.when(F.col("r001") == 1, "Required ID/duplicate check failed"),
            F.when(F.col("r002") == 1, "Referential integrity check failed"),
            F.when(F.col("r003") == 1, "Team integrity check failed"),
            F.when(F.col("r004") == 1, "Cricket sequence check failed"),
            F.when(F.col("r005") == 1, "Runs validation failed"),
            F.when(F.col("r006") == 1, "Wicket consistency check failed"),
            F.when(
                F.col("dq_008_failed") == True,
                F.col("dq_008_reason")
            )
        )
    )

    # --------------------------------------------------------
    # Evidence / routing columns
    # --------------------------------------------------------
    .withColumn(
        "severity",
        F.when(F.col("dq_status") == "FAIL", "ERROR")
         .otherwise("NONE")
    )

    .withColumn(
        "record_key",
        F.concat_ws(
            "|",
            F.col("match_id"),
            F.col("innings_no"),
            F.col("over_no"),
            F.col("ball_seq")
        )
    )

    .withColumn(
        "dq_checked_at",
        F.current_timestamp()
    )

    .withColumn(
        "rework_status",
        F.when(F.col("dq_status") == "FAIL", "OPEN")
         .otherwise("NOT_REQUIRED")
    )
)

display(
    final_deliveries_dq.select(
        "source_record_id",
        "match_id",
        "delivery_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons",
        "severity",
        "record_key",
        "dq_checked_at",
        "rework_status"
    )
)

In [ ]:
# ============================================================
# STEP 8: Final DQ result for deliveries
# ============================================================

final_deliveries_dq = (
    deliveries_q
    .join(
        deliveries_dq_008.select(
            "source_record_id",
            "dq_008_failed",
            "dq_008_reason"
        ),
        "source_record_id",
        "left"
    )
)

# ------------------------------------------------------------
# Treat r001-r006 as STRING rule IDs
# A non-null / non-empty value means that rule failed
# ------------------------------------------------------------

for rule in ["r001", "r002", "r003", "r004", "r005", "r006"]:
    final_deliveries_dq = final_deliveries_dq.withColumn(
        f"{rule}_failed",
        F.when(
            F.col(rule).isNotNull() & (F.trim(F.col(rule)) != ""),
            True
        ).otherwise(False)
    )

# ------------------------------------------------------------
# Combine all failed rule IDs into ONE column
# ------------------------------------------------------------

final_deliveries_dq = final_deliveries_dq.withColumn(
    "dq_rule_ids",
    F.concat_ws(
        ",",
        F.when(F.col("r001_failed"), "DQ-IPL-001"),
        F.when(F.col("r002_failed"), "DQ-IPL-002"),
        F.when(F.col("r003_failed"), "DQ-IPL-003"),
        F.when(F.col("r004_failed"), "DQ-IPL-004"),
        F.when(F.col("r005_failed"), "DQ-IPL-005"),
        F.when(F.col("r006_failed"), "DQ-IPL-006"),
        F.when(F.col("dq_008_failed") == True, "DQ-IPL-008")
    )
)

# ------------------------------------------------------------
# Overall PASS / FAIL
# ------------------------------------------------------------

final_deliveries_dq = final_deliveries_dq.withColumn(
    "dq_status",
    F.when(
        F.col("r001_failed") |
        F.col("r002_failed") |
        F.col("r003_failed") |
        F.col("r004_failed") |
        F.col("r005_failed") |
        F.col("r006_failed") |
        F.coalesce(F.col("dq_008_failed"), F.lit(False)),
        "FAIL"
    ).otherwise("PASS")
)

# ------------------------------------------------------------
# Failure reasons
# ------------------------------------------------------------

final_deliveries_dq = final_deliveries_dq.withColumn(
    "failure_reasons",
    F.concat_ws(
        "; ",
        F.when(F.col("r001_failed"), "Required ID/duplicate check failed"),
        F.when(F.col("r002_failed"), "Referential integrity check failed"),
        F.when(F.col("r003_failed"), "Team integrity check failed"),
        F.when(F.col("r004_failed"), "Cricket sequence check failed"),
        F.when(F.col("r005_failed"), "Runs validation failed"),
        F.when(F.col("r006_failed"), "Wicket consistency check failed"),
        F.when(
            F.col("dq_008_failed") == True,
            F.col("dq_008_reason")
        )
    )
)

# ------------------------------------------------------------
# Evidence columns
# ------------------------------------------------------------

final_deliveries_dq = (
    final_deliveries_dq
    .withColumn(
        "severity",
        F.when(F.col("dq_status") == "FAIL", "ERROR")
         .otherwise("NONE")
    )
    .withColumn(
        "record_key",
        F.concat_ws(
            "|",
            F.col("match_id"),
            F.col("innings_no"),
            F.col("over_no"),
            F.col("ball_seq")
        )
    )
    .withColumn(
        "dq_checked_at",
        F.current_timestamp()
    )
    .withColumn(
        "rework_status",
        F.when(F.col("dq_status") == "FAIL", "OPEN")
         .otherwise("NOT_REQUIRED")
    )
)

display(
    final_deliveries_dq.select(
        "source_record_id",
        "match_id",
        "delivery_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons",
        "severity",
        "record_key",
        "dq_checked_at",
        "rework_status"
    )
)

In [ ]:
final_deliveries_dq.groupBy("dq_status").count().show()

In [ ]:
display(
    final_deliveries_dq
    .filter(F.col("dq_rule_ids").contains(","))
    .select(
        "source_record_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons"
    )
)

Create Trusted Silver + Quarantine Deliveries

In [ ]:
# ============================================================
# STEP 9: Trusted Silver and Quarantine - Deliveries
# ============================================================

# ------------------------------------------------------------
# 1. Trusted Silver: only PASS records
# ------------------------------------------------------------

silver_deliveries = (
    final_deliveries_dq
    .filter(F.col("dq_status") == "PASS")
)

# ------------------------------------------------------------
# 2. Quarantine: only FAIL records
# ------------------------------------------------------------

quarantine_deliveries = (
    final_deliveries_dq
    .filter(F.col("dq_status") == "FAIL")
)

# Check counts
print("Total candidate deliveries :", deliveries.count())
print("Trusted Silver deliveries  :", silver_deliveries.count())
print("Quarantined deliveries     :", quarantine_deliveries.count())

print("\nReconciliation:")
print(
    "Trusted + Quarantine =",
    silver_deliveries.count() + quarantine_deliveries.count()
)

In [ ]:
# ============================================================
# Save Trusted Silver and Quarantine tables
# ============================================================

silver_deliveries.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_deliveries")

quarantine_deliveries.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_deliveries")

print("Tables created successfully.")

In [ ]:
# ============================================================
# Reconciliation check
# ============================================================

candidate_count = deliveries.count()
trusted_count = spark.table(
    "workspace.default.silver_deliveries"
).count()
quarantine_count = spark.table(
    "workspace.default.quarantine_deliveries"
).count()

print("Candidate   :", candidate_count)
print("Trusted     :", trusted_count)
print("Quarantine  :", quarantine_count)
print("Sum         :", trusted_count + quarantine_count)

assert candidate_count == trusted_count + quarantine_count

print("✓ Candidate = Trusted + Quarantine")

Route Matches into Trusted Silver + Quarantine

In [ ]:
# ============================================================
# STEP 10: Trusted Silver + Quarantine - Matches
# ============================================================

# PASS records → Trusted Silver
silver_matches = (
    matches_q
    .filter(F.col("dq_status") == "PASS")
)

# FAIL records → Quarantine
quarantine_matches = (
    matches_q
    .filter(F.col("dq_status") == "FAIL")
)

# Counts
candidate_matches = matches.count()
trusted_matches = silver_matches.count()
failed_matches = quarantine_matches.count()

print("Candidate Matches  :", candidate_matches)
print("Trusted Matches    :", trusted_matches)
print("Quarantine Matches :", failed_matches)
print("Trusted + Quarantine:", trusted_matches + failed_matches)

# Reconciliation
assert candidate_matches == trusted_matches + failed_matches

print("✓ Matches reconciliation passed")

In [ ]:
# ============================================================
# Save Match Trusted + Quarantine tables
# ============================================================

silver_matches.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_matches")

quarantine_matches.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_matches")

print("✓ Match tables saved")

In [ ]:
display(
    spark.table("workspace.default.silver_matches")
    .select(
        "match_id",
        "dq_status",
        "dq_rule_ids"
    )
    .limit(10)
)

display(
    spark.table("workspace.default.quarantine_matches")
    .select(
        "match_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons"
    )
    .limit(10)
)

Route Players into Trusted Silver + Quarantine

In [ ]:
# ============================================================
# STEP 11: Find the player DQ result
# ============================================================

for name, value in list(globals().items()):
    if isinstance(value, DataFrame):
        cols = value.columns
        if "player_id" in cols and "dq_status" in cols:
            print(name, "->", cols)

Create Player DQ result

In [ ]:
# ============================================================
# STEP 11: Player DQ check
# ============================================================

# Check duplicate player IDs
player_dup = (
    players
    .groupBy("player_id")
    .count()
    .filter(F.col("count") > 1)
    .select("player_id")
)

# Build player DQ result
players_q = (
    players
    .join(
        player_dup.withColumn("is_duplicate", F.lit(True)),
        "player_id",
        "left"
    )
    .withColumn(
        "is_duplicate",
        F.coalesce(F.col("is_duplicate"), F.lit(False))
    )
    .withColumn(
        "r001",
        F.when(
            F.col("player_id").isNull() |
            (F.trim(F.col("player_id").cast("string")) == "") |
            F.col("is_duplicate"),
            "DQ-IPL-001"
        )
    )
    .withColumn(
        "dq_rule_ids",
        F.col("r001")
    )
    .withColumn(
        "dq_status",
        F.when(F.col("r001").isNotNull(), "FAIL")
         .otherwise("PASS")
    )
    .withColumn(
        "failure_reasons",
        F.when(
            F.col("player_id").isNull() |
            (F.trim(F.col("player_id").cast("string")) == ""),
            "Player ID is null or blank"
        )
        .when(
            F.col("is_duplicate"),
            "Duplicate player ID"
        )
    )
    .withColumn(
        "severity",
        F.when(F.col("dq_status") == "FAIL", "ERROR")
         .otherwise("NONE")
    )
    .withColumn(
        "record_key",
        F.col("player_id").cast("string")
    )
    .withColumn(
        "dq_checked_at",
        F.current_timestamp()
    )
    .withColumn(
        "rework_status",
        F.when(F.col("dq_status") == "FAIL", "OPEN")
         .otherwise("NOT_REQUIRED")
    )
)

display(
    players_q.select(
        "player_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons"
    )
)


Route Players

In [ ]:
silver_players = (
    players_q
    .filter(F.col("dq_status") == "PASS")
)

quarantine_players = (
    players_q
    .filter(F.col("dq_status") == "FAIL")
)

print("Candidate Players :", players.count())
print("Trusted Players   :", silver_players.count())
print("Quarantine Players:", quarantine_players.count())

assert players.count() == (
    silver_players.count() +
    quarantine_players.count()
)

print("✓ Players reconciliation passed")

In [ ]:
silver_players.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_players")

quarantine_players.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_players")

print("✓ Player tables saved")

Route Venues into Trusted Silver + Quarantine

In [ ]:
# ============================================================
# STEP 12: Venue DQ check
# ============================================================

# Find duplicate venue IDs
venue_dup = (
    venues
    .groupBy("venue_id")
    .count()
    .filter(F.col("count") > 1)
    .select("venue_id")
)

# Build venue DQ result
venues_q = (
    venues
    .join(
        venue_dup.withColumn("is_duplicate", F.lit(True)),
        "venue_id",
        "left"
    )
    .withColumn(
        "is_duplicate",
        F.coalesce(F.col("is_duplicate"), F.lit(False))
    )
    .withColumn(
        "r001",
        F.when(
            F.col("venue_id").isNull() |
            (F.trim(F.col("venue_id").cast("string")) == "") |
            F.col("is_duplicate"),
            "DQ-IPL-001"
        )
    )
    .withColumn(
        "dq_rule_ids",
        F.col("r001")
    )
    .withColumn(
        "dq_status",
        F.when(F.col("r001").isNotNull(), "FAIL")
         .otherwise("PASS")
    )
    .withColumn(
        "failure_reasons",
        F.when(
            F.col("venue_id").isNull() |
            (F.trim(F.col("venue_id").cast("string")) == ""),
            "Venue ID is null or blank"
        )
        .when(
            F.col("is_duplicate"),
            "Duplicate venue ID"
        )
    )
    .withColumn(
        "severity",
        F.when(F.col("dq_status") == "FAIL", "ERROR")
         .otherwise("NONE")
    )
    .withColumn(
        "record_key",
        F.col("venue_id").cast("string")
    )
    .withColumn(
        "dq_checked_at",
        F.current_timestamp()
    )
    .withColumn(
        "rework_status",
        F.when(F.col("dq_status") == "FAIL", "OPEN")
         .otherwise("NOT_REQUIRED")
    )
)

display(
    venues_q.select(
        "venue_id",
        "dq_status",
        "dq_rule_ids",
        "failure_reasons"
    )
)

In [ ]:
silver_venues = (
    venues_q
    .filter(F.col("dq_status") == "PASS")
)

quarantine_venues = (
    venues_q
    .filter(F.col("dq_status") == "FAIL")
)

print("Candidate Venues :", venues.count())
print("Trusted Venues   :", silver_venues.count())
print("Quarantine Venues:", quarantine_venues.count())

assert venues.count() == (
    silver_venues.count() +
    quarantine_venues.count()
)

print("✓ Venues reconciliation passed")

In [ ]:
silver_venues.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_venues")

quarantine_venues.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_venues")

print("✓ Venue tables saved")

Final reconciliation for all 4 entities

In [ ]:
# ============================================================
# STEP 13: Final Candidate vs Trusted + Quarantine Reconciliation
# ============================================================

reconciliation = [
    (
        "matches",
        matches.count(),
        silver_matches.count(),
        quarantine_matches.count()
    ),
    (
        "deliveries",
        deliveries.count(),
        silver_deliveries.count(),
        quarantine_deliveries.count()
    ),
    (
        "players",
        players.count(),
        silver_players.count(),
        quarantine_players.count()
    ),
    (
        "venues",
        venues.count(),
        silver_venues.count(),
        quarantine_venues.count()
    )
]

for entity, candidate, trusted, quarantine in reconciliation:

    total = trusted + quarantine

    print("\n------------------------------")
    print(entity.upper())
    print("------------------------------")
    print("Candidate   :", candidate)
    print("Trusted     :", trusted)
    print("Quarantine  :", quarantine)
    print("Trusted + Q :", total)

    if candidate == total:
        print("✓ RECONCILIATION PASS")
    else:
        print("✗ RECONCILIATION FAIL")

In [ ]:
# ============================================================
# STEP 13B: Trusted ∩ Quarantine check
# ============================================================

checks = [
    (
        "matches",
        silver_matches.select("source_record_id"),
        quarantine_matches.select("source_record_id")
    ),
    (
        "deliveries",
        silver_deliveries.select("source_record_id"),
        quarantine_deliveries.select("source_record_id")
    ),
    (
        "players",
        silver_players.select("source_record_id"),
        quarantine_players.select("source_record_id")
    ),
    (
        "venues",
        silver_venues.select("source_record_id"),
        quarantine_venues.select("source_record_id")
    )
]

for entity, trusted_ids, quarantine_ids in checks:

    overlap = (
        trusted_ids
        .intersect(quarantine_ids)
        .count()
    )

    print(entity.upper(), "overlap =", overlap)

    assert overlap == 0

print("\n✓ Trusted ∩ Quarantine = 0 for all entities")

Controlled Correction + Replay

In [ ]:
# ============================================================
# STEP 14: Controlled Correction + Replay
# ============================================================

# Pick one quarantined delivery
replay_record = (
    quarantine_deliveries
    .select(
        "source_record_id",
        "match_id",
        "innings_no",
        "over_no",
        "ball_seq",
        "total_runs",
        "dq_rule_ids",
        "failure_reasons"
    )
    .limit(1)
)

display(replay_record)

In [ ]:
replay_record.show(truncate=False)

In [ ]:
# ============================================================
# STEP 14B: Inspect parent match teams
# ============================================================

display(
    matches
    .filter(F.col("match_id") == "M2022-001")
    .select(
        "match_id",
        "team_1_id",
        "team_1_name",
        "team_2_id",
        "team_2_name"
    )
)

In [ ]:
display(
    deliveries
    .filter(F.col("source_record_id") == "SRC-DLV-0000001")
    .select(
        "source_record_id",
        "match_id",
        "innings_no",
        "over_no",
        "ball_seq",
        "batting_team_id",
        "bowling_team_id",
        "total_runs"
    )
)

In [ ]:
# ============================================================
# STEP 14C: Controlled correction
# ============================================================

corrected_replay = (
    deliveries
    .filter(
        F.col("source_record_id") == "SRC-DLV-0000001"
    )
    .join(
        matches.select(
            "match_id",
            "team_1_id",
            "team_2_id"
        ),
        "match_id",
        "inner"
    )
    # Correct the team assignment
    .withColumn(
        "batting_team_id",
        F.col("team_1_id")
    )
    .withColumn(
        "bowling_team_id",
        F.col("team_2_id")
    )
    .drop("team_1_id", "team_2_id")
)

display(
    corrected_replay.select(
        "source_record_id",
        "match_id",
        "innings_no",
        "over_no",
        "ball_seq",
        "batting_team_id",
        "bowling_team_id",
        "total_runs"
    )
)

In [ ]:
# ============================================================
# STEP 14D: Replay DQ-IPL-003
# ============================================================

replay_check = (
    corrected_replay
    .join(
        matches.select(
            "match_id",
            "team_1_id",
            "team_2_id"
        ),
        "match_id",
        "inner"
    )
    .withColumn(
        "dq_003_failed",
        ~(
            (
                (
                    F.col("batting_team_id") == F.col("team_1_id")
                ) |
                (
                    F.col("batting_team_id") == F.col("team_2_id")
                )
            )
            &
            (
                (
                    F.col("bowling_team_id") == F.col("team_1_id")
                ) |
                (
                    F.col("bowling_team_id") == F.col("team_2_id")
                )
            )
            &
            (
                F.col("batting_team_id") !=
                F.col("bowling_team_id")
            )
        )
    )
)

display(
    replay_check.select(
        "source_record_id",
        "match_id",
        "batting_team_id",
        "bowling_team_id",
        "dq_003_failed"
    )
)

In [ ]:
# ============================================================
# STEP 15: Week 6 DQ Summary
# ============================================================

summary_data = [
    (
        "matches",
        matches.count(),
        silver_matches.count(),
        quarantine_matches.count()
    ),
    (
        "deliveries",
        deliveries.count(),
        silver_deliveries.count(),
        quarantine_deliveries.count()
    ),
    (
        "players",
        players.count(),
        silver_players.count(),
        quarantine_players.count()
    ),
    (
        "venues",
        venues.count(),
        silver_venues.count(),
        quarantine_venues.count()
    )
]

dq_summary = spark.createDataFrame(
    summary_data,
    [
        "entity",
        "candidate_count",
        "trusted_count",
        "quarantine_count"
    ]
).withColumn(
    "reconciliation_status",
    F.when(
        F.col("candidate_count") ==
        F.col("trusted_count") + F.col("quarantine_count"),
        "PASS"
    ).otherwise("FAIL")
).withColumn(
    "generated_at",
    F.current_timestamp()
)

display(dq_summary)

In [ ]:
# ============================================================
# Save DQ Summary
# ============================================================

dq_summary.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dq_summary")

print("✓ DQ summary saved")

In [ ]:
# ============================================================
# FINAL WEEK 6 VALIDATION
# ============================================================

display(
    dq_summary
    .select(
        "entity",
        "candidate_count",
        "trusted_count",
        "quarantine_count",
        "reconciliation_status"
    )
)